# Lab 17: The Ghost Touch

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 2, *The Time Editor*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-17.ipynb)

**What you will do:** feel the cutaneous rabbit illusion, a phantom middle tap that was never delivered, track where you feel the phantom across a few trials, and watch a masked-language model use a later word to resolve a sentence the same way your brain used a later tap to place the phantom.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 17 a friend tapped your forearm three times, but only two of the taps, both near the
wrist, were real; the middle tap, apparently at mid-forearm, never happened. Your brain used
the timing and location of the third tap (at the elbow) to retroactively place a phantom
sensation between the first two. This is the **cutaneous rabbit illusion**
(Geldard & Sherrick, 1972): **postdiction**, using later information to revise the perceived
timing and location of an earlier event, within a brief window (about 100-200 ms) before
perception is finalised.

## Record your results

Run a few trials, varying which real locations are tapped (e.g. wrist-wrist-elbow, or
wrist-elbow-elbow) and how fast the taps are delivered. For each trial, mark where you felt
the phantom middle tap on a 0-10 scale from wrist (0) to elbow (10), and note whether it
felt like it was roughly at the true midpoint between the two real end taps, or pulled
towards one end.

In [ ]:
# example data: replace with your own trials (positions from 0 = wrist to 10 = elbow)
trials = pd.DataFrame({
    "real_taps": ["wrist, wrist, elbow", "wrist, elbow, elbow", "wrist, wrist, elbow"],
    "tap_interval_ms": [80, 80, 150],
    "felt_phantom_at": [4.5, 6.0, 5.5],
    "true_midpoint": [5.0, 5.0, 5.0],
})
display(trials)
trials.plot.scatter(x="tap_interval_ms", y="felt_phantom_at", s=80, color="steelblue", figsize=(5, 3.5))
plt.axhline(5.0, color="grey", linestyle="--", label="true midpoint (wrist=0, elbow=10)")
plt.xlabel("interval between taps (ms)"); plt.ylabel("felt position of phantom tap"); plt.legend(); plt.show()

## Compare

The lab states that despite only two real taps (both near the wrist) and one at the elbow,
"you will feel three equally spaced taps: wrist, mid-forearm, elbow." That predicts the
phantom should land close to the true midpoint between the first and last real tap
locations. Check how close your own felt positions came to that midpoint.

In [ ]:
trials["error_from_midpoint"] = trials["felt_phantom_at"] - trials["true_midpoint"]
display(trials[["real_taps", "tap_interval_ms", "felt_phantom_at", "error_from_midpoint"]])
print(f"Mean absolute deviation from the predicted midpoint: {trials['error_from_midpoint'].abs().mean():.1f} "
      f"(on a 0-10 wrist-to-elbow scale)")
print("A small deviation supports the lab's description of equal spacing; a large, consistent deviation")
print("in one direction (e.g. always pulled towards the elbow) would suggest the phantom is not placed")
print("exactly at the midpoint for you, which is also a documented finding in some versions of this illusion.")

## The AI side

The lab's AI Connection compares this to **bidirectional** language models, which use words
that come after a target word to help interpret it, exactly as your brain used the third tap
to place the phantom second one. BERT (Devlin, Chang, Lee & Toutanova, 2019, NAACL-HLT) is
trained to fill in a masked word using both the words before and after it. Below, a small
pretrained BERT fills in the gap in the lab's own example sentence, "I saw her duck," which
stays ambiguous until the words after the blank are known.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model="bert-base-uncased")

sentences = [
    "I saw her [MASK].",                       # ambiguous with no following context
    "I saw her [MASK] under the fence.",        # "duck" (verb, to lower the head) becomes likely
    "I saw her [MASK] swim across the pond.",   # "duck" (noun, the bird) becomes likely
]
for s in sentences:
    top = fill_mask(s, top_k=3)
    print(s)
    for t in top:
        print(f"    {t['token_str']!r:10} (score {t['score']:.2f})")
    print()

Notice how the model's top guesses shift once words after the mask are added: it is using
information that comes later in the sentence to reinterpret the meaning of an earlier
position, the same postdictive move your somatosensory cortex made with the phantom tap. The
disanalogy: BERT reads the whole sentence at once, with no sense of "later" arriving after
"earlier" in time; your brain, by contrast, really did receive the third tap after the first
two, and had to hold the earlier taps in an unfinished state for roughly 100-200 ms while it
waited.

## Pool the class data

Pool the class's felt positions of the phantom tap (Step 2 or the smartphone version), using
the same 0 (wrist) to 10 (elbow) scale, to see how consistently people place it near the
midpoint.

In [ ]:
import io
csv_text = """id,felt_phantom_at
P01,4.5
P02,5.5
P03,5.0
P04,6.0
P05,4.0
P06,5.5
"""  # example data: replace with your class CSV, e.g. pd.read_csv("rabbit_class.csv")
class_df = pd.read_csv(io.StringIO(csv_text))
boot = [class_df["felt_phantom_at"].sample(len(class_df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean felt position: {class_df['felt_phantom_at'].mean():.1f} "
      f"(95% bootstrap interval {lo:.1f}-{hi:.1f}, n = {len(class_df)}; true midpoint = 5.0)")
class_df["felt_phantom_at"].plot.hist(bins=6, color="steelblue", alpha=0.7, figsize=(6, 3))
plt.axvline(5.0, color="grey", linestyle="--", label="true midpoint"); plt.legend(); plt.show()

## Questions to think about

1. The lab says the perceived location of the phantom tap is "determined by the location of the last tap." Using your own trial where the real taps were "wrist, elbow, elbow" instead of "wrist, wrist, elbow," did the felt phantom position shift towards the elbow, as the postdiction account predicts?
2. In the BERT demo, the model's answer for "I saw her [MASK]" changes once "under the fence" or "swim across the pond" is added afterwards. Point to the exact moment in the cutaneous rabbit experiment that plays the same role as those extra words.
3. The text says perception accumulates sensory data over "approximately 100-200 milliseconds" before committing to an interpretation. What do you think would happen to the illusion if the delay between the second and third tap were much longer than 200 ms?
4. The Beyond the Lab section argues that postdiction complicates eyewitness testimony about fast-moving events. Based on what you now know about the cutaneous rabbit illusion, what specific claim by a witness (about order, timing, or location) would you treat with most caution?

## Challenge

Try the smartphone vibration version from the lab: two short buzzes with a gap between them,
first both on the inside of your wrist, then one on the inside and one on the outside. Rate,
on the same 0-5 scale as Lab 14's challenge, how strongly you felt a "travelling" vibration
between the two points in each version, and note whether changing the two real locations
changed how convincing the illusion felt.

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-17.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")